In [1]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [5]:
import csv
import datetime as dt
import hashlib
import html as htmllib
import os
import re
import shutil
import sqlite3
import tempfile
import time
import urllib.request
import unicodedata
from urllib.parse import urlparse
from bs4 import BeautifulSoup

FINAL_DB_PATH = "weblog_corpus.db"
TMP_DB_PATH = os.path.join(tempfile.gettempdir(), "weblog_corpus_build.db")
RELEASE_URL = "https://github.com/JasonZeng24/Recovering-and-organizing-Brad-DeLong-s-weblog-archive-pre-2005-/releases/download/V2.1/weblog_corpus.db"
ARCHIVE_FILE = "typepad_archives.txt"
COMPARISON_CSV = "comparison_results.csv"

try:
    import lxml  # noqa: F401
    BS_PARSER = "lxml"
except ImportError:
    BS_PARSER = "html.parser"

TYPEPAD_HOST_PATTERN = re.compile(r"(^|\.)typepad\.com$", re.I)
EDIT_LINK_RE = re.compile(r"typepad\.com/site/blogs", re.I)
BARE_EDIT_URL_RE = re.compile(r"\s*https?://www\.typepad\.com/site/blogs/\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
BLOCK_TAG_RE = re.compile(r"</?(?:p|div|br|blockquote|h[1-6]|li|tr)[^>]*>", re.I)
ENTRY_SEP_RE = re.compile(r"^-{8}\s*$")


def iter_entries(filepath):
    """流式逐篇读取 Movable Type 导出存档"""
    buf = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if ENTRY_SEP_RE.match(line):
                if buf:
                    chunk = "".join(buf).strip()
                    if chunk:
                        yield chunk
                    buf = []
            else:
                buf.append(line)
        if buf:
            chunk = "".join(buf).strip()
            if chunk:
                yield chunk


def parse_full(entry):
    head, *sections = re.split(r"^-----\s*$", entry, flags=re.M)
    meta, categories = {}, []
    for line in head.strip().splitlines():
        if ":" not in line:
            continue
        k, v = line.split(":", 1)
        k, v = k.strip().upper(), v.strip()
        if k in ("CATEGORY", "PRIMARY CATEGORY"):
            if v and v not in categories:
                categories.append(v)
        else:
            meta[k] = v

    fields = {}
    for sec in sections:
        m = re.match(r"\s*(BODY|EXTENDED BODY|EXCERPT|KEYWORDS):[ \t]*\n?(.*)", sec, flags=re.S)
        if m:
            fields[m.group(1)] = m.group(2).strip()

    return {
        "title": meta.get("TITLE", ""),
        "status": meta.get("STATUS", ""),
        "basename": meta.get("BASENAME", ""),
        "url": meta.get("UNIQUE URL", ""),
        "mt_id": meta.get("UNIQUE ID", "") or meta.get("ENTRY ID", ""),
        "date_raw": meta.get("DATE", ""),
        "convert_breaks": meta.get("CONVERT BREAKS", ""),
        "categories": categories,
        "body": fields.get("BODY", ""),
        "extended": fields.get("EXTENDED BODY", ""),
    }


def parse_mt_date(raw):
    for fmt in ("%m/%d/%Y %I:%M:%S %p", "%m/%d/%Y %H:%M:%S"):
        try:
            return dt.datetime.strptime(raw.strip(), fmt)
        except ValueError:
            pass
    return None


def process_markup_fast(raw_html, convert_breaks=""):
    if convert_breaks.lower() == "markdown":
        try:
            import markdown
            raw_html = markdown.markdown(raw_html)
        except ImportError:
            pass

    lower_html = raw_html.lower()
    needs_dom = (
        ("<img" in lower_html)
        or ("typepad.com/site/blogs" in lower_html)
        or ("<script" in lower_html)
        or ("<style" in lower_html)
    )

    if not needs_dom:
        text = BLOCK_TAG_RE.sub("\n\n", raw_html)
        text = HTML_TAG_RE.sub(" ", text)
        text = htmllib.unescape(text)
        text = BARE_EDIT_URL_RE.sub("", text)
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n+", "\n\n", text).strip()
        return text, 0, []

    soup = BeautifulSoup(raw_html, BS_PARSER)
    for t in soup(["script", "style", "noscript"]):
        t.decompose()
    for a in soup.find_all("a", href=EDIT_LINK_RE):
        parent = a.parent
        if parent and parent.name in ("h6", "p", "div") and parent.get_text(strip=True) == a.get_text(strip=True):
            parent.decompose()
        else:
            a.decompose()

    all_imgs_count = 0
    missing_img_urls = []
    for img in soup.find_all("img"):
        src = (img.get("src") or "").strip()
        if not src:
            continue
        all_imgs_count += 1
        host = urlparse(src).netloc
        if (host and TYPEPAD_HOST_PATTERN.search(host)) or not src.startswith("http"):
            if src not in missing_img_urls:
                missing_img_urls.append(src)

    text = htmllib.unescape(soup.get_text("\n\n", strip=True))
    text = BARE_EDIT_URL_RE.sub("", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, all_imgs_count, missing_img_urls


def text_sha256(text):
    norm = " ".join(text.split())
    return hashlib.sha256(norm.encode("utf-8")).hexdigest() if norm else None


def prepare_base_db(t0):
    if os.path.exists(TMP_DB_PATH):
        os.remove(TMP_DB_PATH)

    for candidate in (FINAL_DB_PATH + ".bak", FINAL_DB_PATH):
        if os.path.exists(candidate):
            try:
                c = sqlite3.connect(candidate)
                ok = c.execute("PRAGMA integrity_check;").fetchone()[0] == "ok"
                c.close()
                if ok:
                    if not os.path.exists(FINAL_DB_PATH + ".bak"):
                        shutil.copy2(candidate, FINAL_DB_PATH + ".bak")
                    shutil.copy2(candidate, TMP_DB_PATH)
                    print(f"1/4 已载入完好底库 {candidate} 至工作区 ({time.time()-t0:.1f}s)", flush=True)
                    return
            except sqlite3.DatabaseError:
                pass

    print("1/4 本地无可用原库，正在从 GitHub Release V2.1 下载...", flush=True)
    urllib.request.urlretrieve(RELEASE_URL, TMP_DB_PATH)
    print(f"    下载完成 ({time.time()-t0:.1f}s)", flush=True)


def build_unified_db():
    t0 = time.time()
    if not os.path.exists(ARCHIVE_FILE):
        raise FileNotFoundError(f"当前目录找不到 {ARCHIVE_FILE}，请先将其放入同一文件夹。")

    prepare_base_db(t0)

    conn = sqlite3.connect(TMP_DB_PATH)
    with conn:
        cols = {row[1] for row in conn.execute("PRAGMA table_info(posts)").fetchall()}
        migrations = {
            "file_name": "ALTER TABLE posts ADD COLUMN file_name TEXT",
            "corpus_era": "ALTER TABLE posts ADD COLUMN corpus_era TEXT DEFAULT 'pre-2005'",
            "categories": "ALTER TABLE posts ADD COLUMN categories TEXT",
            "is_canonical_post": "ALTER TABLE posts ADD COLUMN is_canonical_post INTEGER DEFAULT 0",
            "is_unique_corpus": "ALTER TABLE posts ADD COLUMN is_unique_corpus INTEGER DEFAULT 0",
            "exact_post_datetime": "ALTER TABLE posts ADD COLUMN exact_post_datetime TEXT",
            "wayback_url": "ALTER TABLE posts ADD COLUMN wayback_url TEXT",
            "wayback_status": "ALTER TABLE posts ADD COLUMN wayback_status TEXT",
            "mt_entry_id": "ALTER TABLE posts ADD COLUMN mt_entry_id TEXT",
            "permalink": "ALTER TABLE posts ADD COLUMN permalink TEXT",
        }
        for col, ddl in migrations.items():
            if col not in cols:
                conn.execute(ddl)

        # 确保 missing_assets 表存在（与 app.py 兼容）
        conn.execute("""
            CREATE TABLE IF NOT EXISTS missing_assets (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                post_id INTEGER,
                orig_path TEXT,
                kind TEXT,
                status TEXT DEFAULT 'pending',
                wayback_url TEXT
            );
        """)

        # 保证幂等性：清理可能存在的旧 typepad 数据
        conn.execute("DELETE FROM missing_assets WHERE post_id IN (SELECT id FROM posts WHERE corpus_era = 'typepad');")
        conn.execute("DELETE FROM posts WHERE corpus_era = 'typepad';")
        conn.execute("UPDATE posts SET corpus_era = 'pre-2005' WHERE corpus_era IS NULL;")

        seen_rel_paths, seen_permalinks, seen_hashes = set(), set(), set()
        for rpath, perm, body, is_uniq in conn.execute(
            "SELECT rel_path, permalink, body_clean, is_unique_corpus FROM posts"
        ):
            if rpath:
                seen_rel_paths.add(rpath)
            if perm:
                seen_permalinks.add(perm.strip().rstrip("/").lower())
            if is_uniq == 1 and body:
                h = text_sha256(body)
                if h:
                    seen_hashes.add(h)

        wb_cache = {}
        if os.path.exists(COMPARISON_CSV):
            with open(COMPARISON_CSV, encoding="utf-8") as f:
                for r in csv.DictReader(f):
                    if r.get("url") and r.get("capture"):
                        wb_cache[r["url"].strip()] = r["capture"].strip()

        print(f"2/4 正在解析 {ARCHIVE_FILE} 并执行 SHA-256 去重入库...", flush=True)
        cur = conn.cursor()
        inserted_canon = inserted_extra = skipped_dupes = 0
        asset_rows = []

        for idx, raw in enumerate(iter_entries(ARCHIVE_FILE), 1):
            if idx % 3000 == 0:
                print(
                    f"    已扫描 {idx:,} 条 | 新增入库: {inserted_canon + inserted_extra:,} | 耗时: {time.time()-t0:.1f}s",
                    flush=True,
                )

            p = parse_full(raw)
            if not p["status"].lower().startswith("publish"):
                continue

            parsed_dt = parse_mt_date(p["date_raw"])
            is_future = parsed_dt is None or parsed_dt.year > dt.date.today().year
            is_notebook = p["title"].startswith("NOTEBOOK:")

            exact_dt = parsed_dt.strftime("%Y-%m-%d %H:%M:%S") if (parsed_dt and not is_future) else None
            legacy_date = parsed_dt.strftime("%B %d, %Y") if (parsed_dt and not is_future) else "Unknown"

            body_clean_text, total_imgs, missing_img_urls = process_markup_fast(
                f"{p['body']}\n{p['extended']}".strip(), p["convert_breaks"]
            )

            norm_perm = p["url"].strip().rstrip("/").lower()
            content_hash = text_sha256(body_clean_text)
            if (norm_perm and norm_perm in seen_permalinks) or (content_hash and content_hash in seen_hashes):
                skipped_dupes += 1
                continue

            if norm_perm:
                seen_permalinks.add(norm_perm)
            if content_hash:
                seen_hashes.add(content_hash)

            year_folder = str(parsed_dt.year) if (parsed_dt and not is_future) else "undated"
            slug = (p["basename"] or f"entry_{idx}").strip()
            fname = f"{slug}.html"
            rel_path = f"typepad/{year_folder}/{fname}"
            if rel_path in seen_rel_paths:
                fname = f"{slug}_{idx}.html"
                rel_path = f"typepad/{year_folder}/{fname}"
            seen_rel_paths.add(rel_path)

            is_canonical = 0 if (is_notebook or is_future) else 1
            is_unique = 1 if len(body_clean_text) >= 50 else 0
            wb_url = wb_cache.get(p["url"])

            cur.execute("""
                INSERT INTO posts (
                    file_name, rel_path, title, post_date, exact_post_datetime, body_clean,
                    total_imgs, missing_imgs, missing_samples,
                    corpus_era, categories, is_canonical_post, is_unique_corpus,
                    wayback_url, wayback_status, mt_entry_id, permalink
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, 'typepad', ?, ?, ?, ?, ?, ?, ?)
            """, (
                fname,
                rel_path,
                p["title"] or "Untitled TypePad Entry",
                legacy_date,
                exact_dt,
                body_clean_text,
                total_imgs,
                len(missing_img_urls),
                " | ".join(missing_img_urls[:3]),
                "; ".join(p["categories"]),
                is_canonical,
                is_unique,
                wb_url,
                "hit" if wb_url else None,
                p["mt_id"] or None,
                p["url"] or None,
            ))
            new_id = cur.lastrowid
            if is_canonical:
                inserted_canon += 1
            else:
                inserted_extra += 1

            for img_url in missing_img_urls:
                asset_rows.append((new_id, img_url))

        if asset_rows:
            cur.executemany(
                "INSERT INTO missing_assets (post_id, orig_path, kind, status) VALUES (?, ?, 'chart', 'pending')",
                asset_rows,
            )

        print(f"3/4 正在重建 FTS5 全文检索索引 ({time.time()-t0:.1f}s)...", flush=True)
        try:
            cur.execute("INSERT INTO posts_fts(posts_fts) VALUES('rebuild');")
        except sqlite3.OperationalError:
            cur.execute("DELETE FROM posts_fts;")
            cur.execute("INSERT INTO posts_fts(rowid, title, body_clean) SELECT id, title, body_clean FROM posts;")

        total_all, total_uniq, total_canon = cur.execute(
            "SELECT COUNT(*), SUM(is_unique_corpus), SUM(is_canonical_post) FROM posts"
        ).fetchone()

    conn.close()

    print(f"4/4 正在输出最终 {FINAL_DB_PATH} ({time.time()-t0:.1f}s)...", flush=True)
    for suffix in ("", "-wal", "-shm", "-journal"):
        p = FINAL_DB_PATH + suffix
        if os.path.exists(p):
            os.remove(p)
    shutil.move(TMP_DB_PATH, FINAL_DB_PATH)

    print("\n=== 全库合并完成 ===", flush=True)
    print(f"总耗时: {time.time()-t0:.1f}s", flush=True)
    print(f"新增 TypePad 规范博文: {inserted_canon:,} 篇 | 笔记/未定日期: {inserted_extra:,} 篇 | 跳过重叠重复: {skipped_dupes:,} 篇", flush=True)
    print(f"新增待恢复 TypePad 图片资产: {len(asset_rows):,} 条", flush=True)
    print(f"全库总记录: {total_all:,} | 干净去重语料 (is_unique_corpus=1): {total_uniq:,} | 全时期规范博文: {total_canon:,}", flush=True)


if __name__ == "__main__":
    build_unified_db()

1/4 已载入完好底库 weblog_corpus.db.bak 至工作区 (2.7s)
2/4 正在解析 typepad_archives.txt 并执行 SHA-256 去重入库...
    已扫描 3,000 条 | 新增入库: 1,748 | 耗时: 6.8s
    已扫描 6,000 条 | 新增入库: 4,515 | 耗时: 8.5s
3/4 正在重建 FTS5 全文检索索引 (9.5s)...
4/4 正在输出最终 weblog_corpus.db (18.6s)...

=== 全库合并完成 ===
总耗时: 18.6s
新增 TypePad 规范博文: 5,773 篇 | 笔记/未定日期: 31 篇 | 跳过重叠重复: 7 篇
新增待恢复 TypePad 图片资产: 2,258 条
全库总记录: 27,335 | 干净去重语料 (is_unique_corpus=1): 17,526 | 全时期规范博文: 12,306


In [6]:
import json
import os
import re
import sqlite3
from bs4 import BeautifulSoup

DB_PATH = "weblog_corpus.db"
ARCHIVE_FILE = "typepad_archives.txt"
AMBIGUOUS_OUT = "ambiguous_for_llm.jsonl"

ENTRY_SEP_RE = re.compile(r"^-{8}\s*$")
TEACHING_RE = re.compile(r"\b(econ\s*\d+|syllabus|problem\s*set|lecture\s*notes?|course\s*outline|midterm|final\s*exam)\b", re.I)
ROUNDUP_TITLE_RE = re.compile(r"\b(links\s+for|morning\s+coffee|weekend\s+reading|roundup|tweets|link\s+dump|things\s+to\s+read)\b", re.I)
MAIL_PREFIX_RE = re.compile(r"^(from\s+the\s+inbox|reader\s+[\w\s]{1,25}\s+writes|email\s+from|guest\s+post\s+by|forwarded\s+from)\s*:", re.I)
CRITIQUE_TITLE_RE = re.compile(r"(why\s+oh\s+why\s+can't\s+we\s+have\s+a\s+better\s+press\s+corps|dept\.\s+of)", re.I)
SELF_SYNDICATION_RE = re.compile(r"(project\s+syndicate|voxeu|equitable\s+growth|#hoistedfromthearchives)", re.I)
MINIMAL_SIGN_OFF_RE = re.compile(r"(read\s+the\s+whole\s+thing|worth\s+reading|indeed|discuss|go\s+read\s+it)\.?\s*$", re.I)
FIRST_PERSON_RE = re.compile(r"\b(I|I'm|I've|I'd|my|mine|myself)\b")


def extract_typepad_dom_stats(archive_path):
    """从原始 TypePad 导出文件中提取每篇文章的 blockquote 字数占比与超链接数"""
    if not os.path.exists(archive_path):
        return {}

    stats_by_url = {}
    stats_by_slug = {}
    buf = []

    def process_chunk(raw):
        head, *sections = re.split(r"^-----\s*$", raw, flags=re.M)
        meta = {}
        for line in head.strip().splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip().upper()] = v.strip()
        if not meta.get("STATUS", "").lower().startswith("publish"):
            return

        body_parts = []
        for sec in sections:
            m = re.match(r"\s*(BODY|EXTENDED BODY):[ \t]*\n?(.*)", sec, flags=re.S)
            if m:
                body_parts.append(m.group(2).strip())
        html_str = "\n".join(body_parts)
        if not html_str:
            return

        soup = BeautifulSoup(html_str, "html.parser")
        link_count = len(soup.find_all("a"))
        quote_words = 0
        for bq in soup.find_all("blockquote"):
            quote_words += len(bq.get_text(" ", strip=True).split())
            bq.decompose()

        own_text = soup.get_text(" ", strip=True)
        own_words = len(own_text.split())
        total_words = quote_words + own_words
        quote_ratio = (quote_words / total_words) if total_words > 0 else 0.0

        info = {
            "quote_ratio": round(quote_ratio, 4),
            "own_words": own_words,
            "link_count": link_count,
            "own_text_preview": own_text[:500],
        }
        if meta.get("UNIQUE URL"):
            stats_by_url[meta["UNIQUE URL"].strip()] = info
        if meta.get("BASENAME"):
            stats_by_slug[meta["BASENAME"].strip()] = info

    with open(archive_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if ENTRY_SEP_RE.match(line):
                if buf:
                    process_chunk("".join(buf))
                    buf = []
            else:
                buf.append(line)
        if buf:
            process_chunk("".join(buf))

    return {"by_url": stats_by_url, "by_slug": stats_by_slug}


def classify_post(row, dom_info):
    """返回 (post_genre, confidence, needs_llm_review)"""
    title = (row["title"] or "").strip()
    rel_path = (row["rel_path"] or "").strip()
    cats = (row["categories"] or "").strip()
    body = (row["body_clean"] or "").strip()
    is_canon = row["is_canonical_post"]

    words = body.split()
    total_words = len(words)
    head_text = body[:300]
    tail_text = body[-200:] if total_words > 40 else body

    q_ratio = dom_info["quote_ratio"] if dom_info else 0.0
    own_words = dom_info["own_words"] if dom_info else total_words
    link_count = dom_info["link_count"] if dom_info else body.count("http")

    # 1. 教学与元信息
    if TEACHING_RE.search(f"{rel_path} {title} {cats}") or title.startswith("NOTEBOOK:"):
        return "teaching_or_meta", "high", 0

    # 2. 链接汇总 / 微博客
    if ROUNDUP_TITLE_RE.search(title) or (link_count >= 8 and total_words / max(link_count, 1) < 45):
        return "link_roundup", "high", 0

    # 3. 读者来信 / 显式 #noted 剪报
    if MAIL_PREFIX_RE.search(head_text):
        return "clipping_or_mail", "high", 0
    if "#noted" in cats.lower() or "#noted" in tail_text.lower() or "#watch" in cats.lower():
        if own_words < 50:
            return "clipping_or_mail", "high", 0
        return "quote_commentary", "high", 0

    # 4. 基于 HTML <blockquote> 结构的精准切分
    if dom_info and q_ratio > 0:
        if q_ratio >= 0.80 and own_words < 45:
            return "clipping_or_mail", "high", 0
        if q_ratio >= 0.25:
            return "quote_commentary", "high", 0
        return "essay_or_column", "high", 0

    # 5. 无 <blockquote> 时的文本启发式规则（覆盖 Pre-2005 与纯引号引述）
    if CRITIQUE_TITLE_RE.search(title):
        return "quote_commentary", "high", 0

    if SELF_SYNDICATION_RE.search(f"{cats} {head_text}"):
        return "essay_or_column", "high", 0

    # 开头直接是 "Author Name: Title:" 且全文较短、无第一人称展开
    looks_like_citation_lead = bool(re.match(r"^[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3}\s*:\s*", head_text))
    has_first_person = bool(FIRST_PERSON_RE.search(body))

    if looks_like_citation_lead and not has_first_person and total_words < 350:
        return "clipping_or_mail", "medium", 0

    if MINIMAL_SIGN_OFF_RE.search(tail_text) and not has_first_person:
        return "clipping_or_mail", "medium", 0

    # 6. 纯原创长文特征：第一人称丰富且引号/冒号引述较少
    quote_char_ratio = (body.count('"') + body.count("“") + body.count("”")) / max(len(body), 1)
    if has_first_person and quote_char_ratio < 0.008 and total_words >= 120:
        return "essay_or_column", "medium", 0

    # 7. 其余归入模糊集，标记 needs_llm_review = 1
    default_guess = "quote_commentary" if (looks_like_citation_lead or quote_char_ratio >= 0.008) else "essay_or_column"
    return default_guess, "low", 1


def main():
    print("1/3 正在扫描原始 HTML 结构统计 blockquote 占比...")
    dom_maps = extract_typepad_dom_stats(ARCHIVE_FILE)
    by_url = dom_maps.get("by_url", {})
    by_slug = dom_maps.get("by_slug", {})

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cols = {r[1] for r in cur.execute("PRAGMA table_info(posts)")}
    new_cols = {
        "post_genre": "ALTER TABLE posts ADD COLUMN post_genre TEXT",
        "quote_ratio": "ALTER TABLE posts ADD COLUMN quote_ratio REAL DEFAULT 0.0",
        "own_words": "ALTER TABLE posts ADD COLUMN own_words INTEGER DEFAULT 0",
        "needs_llm_review": "ALTER TABLE posts ADD COLUMN needs_llm_review INTEGER DEFAULT 0",
    }
    for col, ddl in new_cols.items():
        if col not in cols:
            cur.execute(ddl)

    rows = cur.execute("""
        SELECT id, file_name, rel_path, title, categories, body_clean,
               is_canonical_post, is_unique_corpus, permalink
        FROM posts
    """).fetchall()

    print(f"2/3 正在为 {len(rows):,} 条记录判定体裁...")
    updates = []
    ambiguous_records = []
    genre_counts = {}

    for r in rows:
        perm = (r["permalink"] or "").strip()
        slug = os.path.splitext(r["file_name"] or "")[0]
        dom_info = by_url.get(perm) or by_slug.get(slug)

        genre, conf, needs_review = classify_post(r, dom_info)
        q_ratio = dom_info["quote_ratio"] if dom_info else 0.0
        o_words = dom_info["own_words"] if dom_info else len((r["body_clean"] or "").split())

        updates.append((genre, q_ratio, o_words, needs_review, r["id"]))
        if r["is_unique_corpus"] == 1:
            genre_counts[genre] = genre_counts.get(genre, 0) + 1
            if needs_review == 1:
                body_str = r["body_clean"] or ""
                ambiguous_records.append({
                    "id": r["id"],
                    "title": r["title"],
                    "categories": r["categories"],
                    "head": body_str[:1200],
                    "tail": body_str[-800:] if len(body_str) > 1200 else "",
                })

    cur.executemany(
        "UPDATE posts SET post_genre = ?, quote_ratio = ?, own_words = ?, needs_llm_review = ? WHERE id = ?",
        updates,
    )
    conn.commit()
    conn.close()

    with open(AMBIGUOUS_OUT, "w", encoding="utf-8") as f:
        for rec in ambiguous_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("\n=== 去重语料（17,526 篇）体裁分布统计 ===")
    for g, cnt in sorted(genre_counts.items(), key=lambda x: -x[1]):
        print(f"  {g:<20}: {cnt:,} 篇")
    print(f"\n3/3 已导出 {len(ambiguous_records):,} 条低置信度样本至 {AMBIGUOUS_OUT}，可直接送入本地 Ollama 复核。")


if __name__ == "__main__":
    main()

1/3 正在扫描原始 HTML 结构统计 blockquote 占比...
2/3 正在为 27,335 条记录判定体裁...

=== 去重语料（17,526 篇）体裁分布统计 ===
  essay_or_column     : 11,590 篇
  quote_commentary    : 4,445 篇
  clipping_or_mail    : 755 篇
  link_roundup        : 507 篇
  teaching_or_meta    : 229 篇

3/3 已导出 3,675 条低置信度样本至 ambiguous_for_llm.jsonl，可直接送入本地 Ollama 复核。


In [ ]:
import json
import os
import sqlite3
import time
import urllib.request

DB_PATH = "weblog_corpus.db"
AMBIGUOUS_IN = "ambiguous_for_llm.jsonl"
CHECKPOINT_OUT = "ollama_reviewed.jsonl"
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:7b"

VALID_GENRES = {
    "essay_or_column",
    "quote_commentary",
    "clipping_or_mail",
    "link_roundup",
    "teaching_or_meta",
}

SYSTEM_PROMPT = """You are a corpus classifier for economist J. Bradford DeLong's blog.
Classify each post into EXACTLY ONE of these 5 genres based on its title, opening, and closing:
- "essay_or_column": Original essay, economic analysis, or syndicated column written primarily by DeLong himself (quotes < 30%).
- "quote_commentary": Excerpts/quotes from another author (30%-80%) paired with substantive commentary, critique, or rebuttal by DeLong.
- "clipping_or_mail": Pure bookmark/excerpt from another writer, reader email ("From the inbox"), or guest post with minimal/no commentary by DeLong (e.g., ending with "Read the whole thing", "Indeed", or "#noted").
- "link_roundup": List of multiple external links, daily reading dumps, or tweet roundups.
- "teaching_or_meta": Course syllabus, reading list, lecture notes, problem set, or blog housekeeping notice.

Respond ONLY with valid JSON: {"genre": "<one_of_the_5_labels>"}"""


def classify_with_ollama(rec):
    user_prompt = (
        f"Title: {rec.get('title', '')}\n"
        f"Categories: {rec.get('categories', '')}\n"
        f"--- OPENING ---\n{rec.get('head', '')}\n"
        f"--- CLOSING ---\n{rec.get('tail', '')}"
    )
    payload = {
        "model": MODEL_NAME,
        "system": SYSTEM_PROMPT,
        "prompt": user_prompt,
        "format": "json",
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 32},
    }
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        res = json.loads(resp.read().decode("utf-8"))
        parsed = json.loads(res.get("response", "{}"))
        genre = str(parsed.get("genre", "")).strip()
        return genre if genre in VALID_GENRES else None


def main():
    if not os.path.exists(AMBIGUOUS_IN):
        raise FileNotFoundError(f"找不到 {AMBIGUOUS_IN}")

    # 1. 加载已完成断点
    done = {}
    if os.path.exists(CHECKPOINT_OUT):
        with open(CHECKPOINT_OUT, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    obj = json.loads(line)
                    done[obj["id"]] = obj["genre"]
        print(f"已加载历史进度：{len(done):,} 条已完成", flush=True)

    with open(AMBIGUOUS_IN, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    todo = [r for r in records if r["id"] not in done]
    print(f"待复核样本：{len(todo):,} 条（使用模型: {MODEL_NAME}）...", flush=True)

    t0 = time.time()
    errors = 0
    with open(CHECKPOINT_OUT, "a", encoding="utf-8") as out_f:
        for i, rec in enumerate(todo, 1):
            try:
                genre = classify_with_ollama(rec)
                if genre:
                    done[rec["id"]] = genre
                    out_f.write(json.dumps({"id": rec["id"], "genre": genre}) + "\n")
                else:
                    errors += 1
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  [警告] ID {rec['id']} 请求失败: {e}", flush=True)

            if i % 100 == 0 or i == len(todo):
                out_f.flush()
                elapsed = time.time() - t0
                rate = i / max(elapsed, 0.1)
                eta = (len(todo) - i) / max(rate, 0.1)
                print(
                    f"  进度: {i:,}/{len(todo):,} ({i/len(todo):.1%}) | "
                    f"速度: {rate:.1f} 条/秒 | 预计剩余: {eta:.0f}s",
                    flush=True,
                )

    # 2. 批量写回 SQLite 数据库
    print("\n正在将复核结果写回 weblog_corpus.db ...", flush=True)
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    updates = [(g, pid) for pid, g in done.items()]
    cur.executemany(
        "UPDATE posts SET post_genre = ?, needs_llm_review = 0 WHERE id = ?",
        updates,
    )
    conn.commit()

    print("\n=== 最终去重语料（17,526 篇）体裁分布 ===")
    for genre, cnt in cur.execute("""
        SELECT post_genre, COUNT(*) AS c
        FROM posts
        WHERE is_unique_corpus = 1
        GROUP BY post_genre
        ORDER BY c DESC
    """):
        print(f"  {genre:<20}: {cnt:,} 篇")
    conn.close()


if __name__ == "__main__":
    main()

In [7]:
import concurrent.futures
import json
import os
import re
import shutil
import sqlite3
import threading
import time
import urllib.request

DB_PATH = "weblog_corpus.db"
BACKUP_PATH = "weblog_corpus.db.before_llm_clean.bak"
AUDIT_LOG = "llm_clean_audit.jsonl"
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:7b"
MAX_WORKERS = 3

# 触发大模型检查的“疑似污染”信号（宽松触发，宁可多查几百篇也不漏网）
SUSPECT_RE = re.compile(
    r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffd]|"
    r"Tranfer path:|Transfer path:|Subject:\s*Homework|"
    r"[^\w\s.,;:'\"!?()\[\]{}\-–—/\\@#$%&*+=<>|~`^‘’“”…£€¥¢§°±×÷]{3,}"
)

SYSTEM_PROMPT = """You are a strict archival text restorer for economist J. Bradford DeLong's blog.
Your job is to clean corrupted text while preserving 100% of authentic human-written prose VERBATIM.

RULES:
1. If the entire input is unreadable binary garbage, random symbols, or spam/email routing headers with no real blog prose, output EXACTLY:
   {"status": "PURE_GARBAGE", "clean_text": ""}
2. If the input contains real English/Chinese blog writing mixed with garbled encoding symbols, binary chunks, or spam headers:
   - Remove ONLY the garbled binary/symbol sequences and spam routing headers.
   - Fix obvious broken encoding characters (like stray replacement boxes) if the surrounding word is clear.
   - Keep ALL legitimate paragraphs, quotes, and sentences WORD-FOR-WORD. Do NOT summarize, paraphrase, or shorten valid prose.
   - Output JSON: {"status": "REPAIRED", "clean_text": "<the verbatim cleaned text>"}

Respond ONLY with valid JSON."""


def is_suspect_doc(body: str) -> bool:
    if not body or len(body.strip()) < 25:
        return True
    if SUSPECT_RE.search(body):
        return True
    # 检查是否存在异常超长无空格乱码词
    no_url = re.sub(r"https?://\S+", "", body)
    words = no_url.split()
    if words and max(len(w) for w in words) > 55:
        return True
    return False


def clean_chunk_with_ollama(text_chunk: str):
    payload = {
        "model": MODEL_NAME,
        "system": SYSTEM_PROMPT,
        "prompt": f"Clean the following archival text:\n\n{text_chunk}",
        "format": "json",
        "stream": False,
        "keep_alive": "20m",
        "options": {
            "temperature": 0.0,
            "num_ctx": 8192,
            "num_predict": 4096,
        },
    }
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode("utf-8"))
        parsed = json.loads(res.get("response", "{}"))
        status = parsed.get("status", "")
        clean_text = str(parsed.get("clean_text", "")).strip()
        return status, clean_text


def process_record(rec):
    pid, title, rel_path, body = rec
    # 若文章极长，仅对包含疑似乱码的段落窗口送入模型，避免截断或过度耗时
    paragraphs = re.split(r"\n\s*\n", body)
    if len(body) > 6000 and len(paragraphs) > 1:
        rebuilt = []
        any_repaired = False
        for p in paragraphs:
            if is_suspect_doc(p):
                st, cleaned = clean_chunk_with_ollama(p[:5500])
                if st == "REPAIRED" and cleaned:
                    rebuilt.append(cleaned)
                    any_repaired = True
                elif st == "PURE_GARBAGE":
                    any_repaired = True
                else:
                    rebuilt.append(p)
            else:
                rebuilt.append(p)
        final_text = "\n\n".join(rebuilt).strip()
        if len(final_text.split()) < 25:
            return pid, rel_path, "PURE_GARBAGE", body, ""
        return pid, rel_path, ("REPAIRED" if any_repaired else "UNCHANGED"), body, final_text

    # 常规长度文档整篇送入模型精确修复
    st, cleaned = clean_chunk_with_ollama(body[:6000])
    if st == "PURE_GARBAGE" or len(cleaned.split()) < 25:
        return pid, rel_path, "PURE_GARBAGE", body, ""
    if st == "REPAIRED" and cleaned:
        return pid, rel_path, "REPAIRED", body, cleaned
    return pid, rel_path, "UNCHANGED", body, body


def main():
    print(f"1/4 正在创建安全备份: {BACKUP_PATH} ...", flush=True)
    shutil.copy2(DB_PATH, BACKUP_PATH)

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    rows = cur.execute("SELECT id, title, rel_path, body_clean FROM posts").fetchall()

    suspects = [r for r in rows if is_suspect_doc(r[3] or "")]
    print(f"2/4 全库 {len(rows):,} 条记录中，筛出疑似含乱码/污染样本 {len(suspects):,} 条，开始调用 {MODEL_NAME} 清洗...", flush=True)

    repaired_updates = []
    garbage_ids = []
    t0 = time.time()
    done_cnt = 0
    lock = threading.Lock()

    with open(AUDIT_LOG, "w", encoding="utf-8") as log_f:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process_record, r) for r in suspects]
            for fut in concurrent.futures.as_completed(futures):
                try:
                    pid, rel_path, status, orig_body, new_body = fut.result()
                    with lock:
                        done_cnt += 1
                        if status == "PURE_GARBAGE":
                            garbage_ids.append((pid,))
                            log_f.write(json.dumps({
                                "id": pid, "rel_path": rel_path, "action": "ISOLATED_GARBAGE",
                                "sample": orig_body[:250]
                            }, ensure_ascii=False) + "\n")
                        elif status == "REPAIRED" and new_body != orig_body:
                            repaired_updates.append((new_body, pid))
                            log_f.write(json.dumps({
                                "id": pid, "rel_path": rel_path, "action": "REPAIRED_TEXT",
                                "before_sample": orig_body[:250],
                                "after_sample": new_body[:250]
                            }, ensure_ascii=False) + "\n")

                        if done_cnt % 25 == 0 or done_cnt == len(suspects):
                            log_f.flush()
                            rate = done_cnt / max(time.time() - t0, 0.1)
                            eta = (len(suspects) - done_cnt) / max(rate, 0.1)
                            print(f"  进度: {done_cnt}/{len(suspects)} | 速度: {rate:.1f} 条/秒 | 预计剩余: {eta:.0f}s", flush=True)
                except Exception as e:
                    with lock:
                        done_cnt += 1
                        print(f"  [跳过异常] {e}", flush=True)

    print("3/4 正在写回修复结果并重建 FTS5 索引...", flush=True)
    if repaired_updates:
        cur.executemany("UPDATE posts SET body_clean = ? WHERE id = ?", repaired_updates)
    if garbage_ids:
        cur.executemany(
            "UPDATE posts SET is_unique_corpus = 0, is_canonical_post = 0, post_genre = 'corrupt_binary' WHERE id = ?",
            garbage_ids,
        )
    cur.execute("INSERT INTO posts_fts(posts_fts) VALUES('rebuild');")
    conn.commit()
    conn.close()

    print(f"\n4/4 清洗完成！")
    print(f"  - LLM 成功剥离乱码并挽救正文: {len(repaired_updates):,} 篇")
    print(f"  - 确认为纯二进制乱码并隔离  : {len(garbage_ids):,} 篇")
    print(f"  - 前后对照审计日志已保存至  : {AUDIT_LOG}")


if __name__ == "__main__":
    main()

1/4 正在创建安全备份: weblog_corpus.db.before_llm_clean.bak ...
2/4 全库 27,335 条记录中，筛出疑似含乱码/污染样本 826 条，开始调用 qwen2.5:7b 清洗...


KeyboardInterrupt: 